In [1]:
import kaggle

# Download dataset
!kaggle datasets download -d snehaanbhawal/resume-dataset -p ./data

# Unzip
import zipfile
with zipfile.ZipFile("./data/resume-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("./data")


Dataset URL: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
License(s): CC0-1.0
resume-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
import os

print(os.listdir("./data"))


['data', 'Resume', 'resume-dataset.zip']


In [3]:
import os
print(os.listdir("./data/Resume"))


['Resume.csv', 'Resume_extended.csv']


In [8]:
print(df["Category"].unique())


NameError: name 'df' is not defined

In [4]:
import os
import pandas as pd

# Ensure the folder exists
os.makedirs("./data/Resume", exist_ok=True)

# Load your original dataset
df = pd.read_csv("./data/Resume/Resume.csv")

# Synthetic "Other" resumes
other_samples = [
    "Mechanical engineer with 5 years of experience in HVAC design, AutoCAD, and manufacturing processes.",
    "Civil engineer skilled in construction management, site supervision, and structural design.",
    "Automobile engineer with hands-on experience in vehicle design, testing, and maintenance.",
    "Registered nurse with expertise in patient care, hospital operations, and emergency response.",
    "Doctor with specialization in internal medicine and clinical diagnosis.",
    "HR professional with experience in recruitment, employee relations, and payroll management.",
    "Administrative assistant skilled in office coordination, documentation, and scheduling.",
    "High school teacher specializing in mathematics, physics, and classroom management.",
    "University lecturer in business administration with research background.",
    "Accountant with expertise in auditing, tax filing, and financial reporting.",
    "Banking professional with experience in loan processing, risk analysis, and customer service.",
    "Graphic designer with skills in Adobe Photoshop, Illustrator, and creative branding.",
    "Journalist with experience in news reporting, content writing, and media communication.",
    "Chef with 10 years of experience in kitchen operations and menu design.",
    "Fitness trainer with certifications in personal training and nutrition planning.",
]

# Wrap into DataFrame
other_df = pd.DataFrame({
    "Resume_str": other_samples,
    "Category": "Other"
})

# Merge and save
df_extended = pd.concat([df, other_df], ignore_index=True)
df_extended.to_csv("./data/Resume/Resume_extended.csv", index=False)

print("✅ Extended dataset created: ./data/Resume/Resume_extended.csv")


✅ Extended dataset created: ./data/Resume/Resume_extended.csv


In [5]:
import pandas as pd

# Load dataset
df = pd.read_csv("./data/Resume/Resume_extended.csv")

# Add synthetic non-software resumes
other_samples = [
    "Mechanical engineer with 5 years of experience in HVAC design, AutoCAD, and manufacturing processes.",
    "Civil engineer skilled in construction management, site supervision, and structural design.",
    "Nurse with expertise in patient care, hospital operations, and emergency response.",
    "HR professional with experience in recruitment, employee relations, and payroll management.",
    "Teacher specializing in mathematics and classroom management.",
    "Accountant with expertise in auditing, tax filing, and financial reporting."
]

other_df = pd.DataFrame({
    "Resume_str": other_samples,
    "Category": "Other"
})

# Merge into main dataset
df = pd.concat([df, other_df], ignore_index=True)


# Keep IT/software + "Other"
software_df = df[df["Category"].isin(
    ["INFORMATION-TECHNOLOGY", "ENGINEERING", "CONSULTANT", "Other"]
)].copy()

# Map categories
role_map = {
    "INFORMATION-TECHNOLOGY": "Software Engineer",
    "ENGINEERING": "Developer",
    "CONSULTANT": "Data Analyst",
    "Other": "Other"   # 👈 add this
}
software_df["RefinedRole"] = software_df["Category"].map(role_map)

print("Unique roles:", software_df["RefinedRole"].unique())
print("Counts per role:")
print(software_df["RefinedRole"].value_counts())


Unique roles: ['Software Engineer' 'Data Analyst' 'Developer' 'Other']
Counts per role:
RefinedRole
Software Engineer    120
Developer            118
Data Analyst         115
Other                 21
Name: count, dtype: int64


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# === Clean text ===
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Cleaned_Resume"] = df["Resume_str"].apply(clean_text)

software_df["Cleaned_Resume"] = software_df["Resume_str"].apply(clean_text)

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(software_df["Cleaned_Resume"])
y = software_df["RefinedRole"]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

print("🔄 Training Random Forest model...")
clf.fit(X_train, y_train)   # ✅ This line actually trains the model

# Evaluate
y_pred = clf.predict(X_test)
print("\n=== Model Performance (Software Roles) ===")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

# Save model and vectorizer
joblib.dump(clf, "software_roles_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("\n✅ Model and vectorizer saved successfully!")


🔄 Training Random Forest model...

=== Model Performance (Software Roles) ===
                   precision    recall  f1-score   support

     Data Analyst       0.91      0.91      0.91        23
        Developer       0.92      0.96      0.94        24
            Other       1.00      1.00      1.00         4
Software Engineer       0.91      0.88      0.89        24

         accuracy                           0.92        75
        macro avg       0.94      0.94      0.94        75
     weighted avg       0.92      0.92      0.92        75

Accuracy: 0.92

✅ Model and vectorizer saved successfully!


In [12]:
import joblib

# Save trained models & vectorizer
joblib.dump(clf, "software_roles_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")


['vectorizer.pkl']

In [13]:
with open("app.py", "w", encoding="utf-8") as f:
    f.write("""
import streamlit as st
import joblib
import re

# Load models & vectorizer
l1_model = joblib.load("l1_model.pkl")
l2_model = joblib.load("l2_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

# Cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\\s+', ' ', text)
    return text

st.title("Resume Job Role Matcher")
st.write("Paste a resume below and get predictions!")

resume_text = st.text_area("Paste Resume Text Here:")

if st.button("Predict Job Role"):
    if resume_text.strip():
        cleaned = clean_text(resume_text)
        features = vectorizer.transform([cleaned])

        l1_pred = l1_model.predict(features)[0]
        l2_pred = l2_model.predict(features)[0]

        st.success(f"L1 Logistic Regression Prediction: {l1_pred}")
        st.success(f"L2 Logistic Regression Prediction: {l2_pred}")
    else:
        st.warning("Please paste a resume before clicking Predict.")
""")

print("✅ app.py file created successfully!")


✅ app.py file created successfully!


In [12]:
%pip install pyngrok


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [3]:
from pyngrok import ngrok, conf

# Replace with your actual token
authtoken = "32YfO5UM4Z5CIq2HSxoixPZm0g4_65YmAwAcfHGQmmrAEkh5f"
ngrok.set_auth_token(authtoken)

print("✅ Ngrok Authtoken saved!")


✅ Ngrok Authtoken saved!


In [5]:
import subprocess
from pyngrok import ngrok

# Kill old tunnels
ngrok.kill()

# Start Streamlit app
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)

# Get public URL
public_url = ngrok.connect(8501)
print("✅ Your app is running here:", public_url)


✅ Your app is running here: NgrokTunnel: "https://4b0c6abf37f7.ngrok-free.app" -> "http://localhost:8501"


In [14]:
pip install pdfplumber


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   -------------- ------------------------- 2.1/5.6 MB 14.5 MB/s eta 0:00:01
   ----------------------------- ---------- 4.2/5.6 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------  5.5/5.6 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 9.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   --------------------- ------------------ 1.6/2.9 MB 27.2 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.9 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 8.2 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [pypdfium2]
   ---------------------------------------- 0/3 [pypdfium2]
   ---------------------------------------- 0/3 [pypdfium2]
   ---------------------------------------- 0/3 [pypdfium2]
   -----